### Build Results Fact

1. Read silver results table
2. Read silver sprints table
3. Add new column session_type with values RACE or SPRINT
4. UNION results and sprints
5. Derive additional columns:
    * is_win → Indicates that the driver won the race
    * is_podium → Indicates that the driver scored a podium result (1, 2, 3)
    * has_points → Indicates that the driver has scored points
6. Write the transformed data to gold fact_session_results table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table =F"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

### Step 1 - Read Source Table
- silver.results
- silver.sprints

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
        .withColumn("session_type", F.lit("Race"))
        .drop("race_date", "race_name", "timestamp", "source_file")
)

In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
        .withColumn("session_type", F.lit("Sprint"))
        .drop("race_date", "race_name", "timestamp", "source_file")
)

#### Step 2 - UNION `results` and `sprints`

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)


### Step 3 - Add derived columns
- 1. is_win -> Indicates that the driver own the race
- 2. is_podium -> indicates that the driver scored a podium results(1,2,3)
- 3. has_points -> Indicate that the driver has scored points

In [0]:
    results_Final_df = (
        results_sprints_df
            .withColumn("is_win", F.col("finish_position") == 1)
            .withColumn("is_podium", F.col("finish_position").between(1, 3))
            .withColumn("has_points", F.col("points") > 0)
    )   


In [0]:
display(results_Final_df.filter("season = 2025"))

round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,session_type,is_win,is_podium,has_points
1,2025,williams,albon,6,57,23,10.0,5,5,Finished,Race,false,false,true
1,2025,aston_martin,alonso,12,32,14,0.0,17,R,Retired,Race,false,false,false
1,2025,mercedes,antonelli,16,57,12,12.0,4,4,Finished,Race,false,false,true
1,2025,haas,bearman,20,57,87,0.0,14,14,Finished,Race,false,false,false
1,2025,sauber,bortoleto,15,45,5,0.0,16,R,Retired,Race,false,false,false
1,2025,alpine,doohan,14,0,7,0.0,19,R,Retired,Race,false,false,false
1,2025,alpine,gasly,9,57,10,0.0,11,11,Finished,Race,false,false,false
1,2025,rb,hadjar,11,0,6,0.0,20,R,Retired,Race,false,false,false
1,2025,ferrari,hamilton,8,57,44,1.0,10,10,Finished,Race,false,false,true
1,2025,sauber,hulkenberg,17,57,27,6.0,7,7,Finished,Race,false,false,true


In [0]:
(
    results_Final_df
        .write
        .mode("overwrite")
        .format("delta")
        .option('mergeSchema', 'true').saveAsTable(target_table)
)       

In [0]:
display(spark.read.table(target_table))

round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,session_type,is_win,is_podium,has_points
null,null,ferrari,phil_hill,1,50,20,8.0,1,1,Finished,Race,true,true,true
9,null,porsche,barth,12,47,24,0.0,7,7,+3 Laps,Race,false,false,false
9,null,cooper-castellotti,cabianca,4,48,2,3.0,4,4,+2 Laps,Race,false,false,true
9,null,porsche,herrmann,10,47,26,1.0,6,6,+3 Laps,Race,false,false,true
9,null,ferrari,trips,6,48,22,2.0,5,5,+2 Laps,Race,false,false,true
1,1950,maserati,bira,5,49,21,0.0,14,R,Out of fuel,Race,false,false,false
1,1950,lago,cabantous,6,68,14,3.0,4,4,+2 Laps,Race,false,false,true
1,1950,maserati,chiron,11,26,19,0.0,18,R,Clutch,Race,false,false,false
1,1950,lago,claes,21,64,18,0.0,11,11,+6 Laps,Race,false,false,false
1,1950,alta,crossley,17,43,24,0.0,16,R,Transmission,Race,false,false,false
